# PPO Pipeline A (27-trial grid)

27-trial grid {timesteps × learning_rate × seed} on the fixed train/val/test split. Writes the PPO rows of pipeline_a_validation_grid.csv, the PPO entry of pipeline_a_best_configs.json, and the PPO portion of pipeline_a_{val,test}_results.csv plus the daily-returns CSVs.


# Imports


In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from itertools import product

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

warnings.filterwarnings('ignore')


# Configuration


In [ ]:
TIMESTEP_GRID = [250_000, 500_000, 1_000_000]
PPO_LR_GRID = [1e-4, 3e-4, 1e-3]
SEEDS = [42, 123, 456]

EPISODE_LENGTH = 252
TRANSACTION_COST = 0.001
REWARD_SCALE = 100

TRAIN_DATA_PATH = 'files/train_data_v3.csv.gz'
VAL_DATA_PATH = 'files/val_data_v3.csv.gz'
TEST_DATA_PATH = 'files/test_data_v3.csv.gz'
SCALER_PARAMS_PATH = 'results/scaler_params.csv'

RESULTS_CSV = 'results/pipeline_a_validation_grid.csv'
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)


# Environment


In [ ]:
class PortfolioEnv(gym.Env):
    """Continuous-action portfolio environment for PPO."""
    
    metadata = {"render_modes": []}
    TRANSACTION_COST = TRANSACTION_COST
    
    def __init__(self, df, episode_length=252, mode='train'):
        super().__init__()
        self.mode = mode
        self.episode_length = episode_length
        
        self.dates = sorted(df['date'].unique())
        self.stocks = sorted(df['PERMNO'].unique())
        self.n_assets = len(self.stocks)
        
        self.feature_cols = ["PRC","VOL","RET","SHROUT","OPENPRC","vix_index",
                             "RSI","MACD","BB_PCT","ATR","rolling_vol_60",
                             "SMA_20","SMA_50","log_vol"]
        self.n_features = len(self.feature_cols)
        
        self.obs_matrix = np.zeros((len(self.dates), self.n_assets, self.n_features))
        
        if 'raw_RET' in df.columns:
            ret_col = 'raw_RET'
        else:
            ret_col = 'RET'
            print("  WARNING: Using normalized RET for rewards")
        
        self.ret_matrix = np.zeros((len(self.dates), self.n_assets))
        
        date_to_idx = {d: i for i, d in enumerate(self.dates)}
        stock_to_idx = {s: i for i, s in enumerate(self.stocks)}
        
        for _, row in df.iterrows():
            d_idx = date_to_idx[row['date']]
            s_idx = stock_to_idx[row['PERMNO']]
            self.obs_matrix[d_idx, s_idx] = row[self.feature_cols].values.astype(np.float64)
            self.ret_matrix[d_idx, s_idx] = row[ret_col]
        
        # FIX: Clean NaN/Inf from observation matrix
        nan_count = np.isnan(self.obs_matrix).sum()
        inf_count = np.isinf(self.obs_matrix).sum()
        if nan_count > 0 or inf_count > 0:
            print(f"  WARNING: Found {nan_count} NaN and {inf_count} Inf values in obs_matrix — replacing with 0")
            self.obs_matrix = np.nan_to_num(self.obs_matrix, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Clip extreme values to prevent gradient explosion
        obs_clip = 10.0
        clipped = np.sum(np.abs(self.obs_matrix) > obs_clip)
        if clipped > 0:
            pct = clipped / self.obs_matrix.size * 100
            print(f"  Clipping {clipped} values ({pct:.2f}%) outside [-{obs_clip}, {obs_clip}]")
            self.obs_matrix = np.clip(self.obs_matrix, -obs_clip, obs_clip)
        
        # Also clean return matrix
        nan_ret = np.isnan(self.ret_matrix).sum()
        if nan_ret > 0:
            print(f"  WARNING: Found {nan_ret} NaN values in ret_matrix — replacing with 0")
            self.ret_matrix = np.nan_to_num(self.ret_matrix, nan=0.0)
        
        # Verify no remaining issues
        assert not np.isnan(self.obs_matrix).any(), "obs_matrix still has NaN!"
        assert not np.isinf(self.obs_matrix).any(), "obs_matrix still has Inf!"
        
        obs_dim = self.n_assets * self.n_features + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Box(-1, 1, shape=(self.n_assets,), dtype=np.float32)
        
        self.weights = np.ones(self.n_assets) / self.n_assets
        self.current_step = 0
        self.start_step = 0
    
    def _get_obs(self):
        features = self.obs_matrix[self.current_step].flatten()
        obs = np.concatenate([features, self.weights]).astype(np.float32)
        # Final safety check
        obs = np.nan_to_num(obs, nan=0.0, posinf=0.0, neginf=0.0)
        return obs
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if self.mode == 'train':
            max_start = len(self.dates) - self.episode_length - 1
            self.start_step = self.np_random.integers(0, max(1, max_start))
        else:
            self.start_step = 0
        self.current_step = self.start_step
        self.weights = np.ones(self.n_assets) / self.n_assets
        return self._get_obs(), {}
    
    def step(self, action):
        # Softmax to get valid portfolio weights
        action = np.nan_to_num(action, nan=0.0)  # safety
        exp_a = np.exp(action - np.max(action))
        weights = exp_a / (exp_a.sum() + 1e-8)  # epsilon to prevent div by zero
        
        turnover = np.sum(np.abs(weights - self.weights))
        tc = self.TRANSACTION_COST * turnover
        
        self.current_step += 1
        returns = self.ret_matrix[self.current_step]
        portfolio_return = np.dot(weights, returns) - tc
        reward = portfolio_return * REWARD_SCALE
        
        # Clip reward to prevent extreme gradients
        reward = np.clip(reward, -10, 10)
        
        self.weights = weights
        steps_taken = self.current_step - self.start_step
        done = (steps_taken >= self.episode_length) or \
               (self.current_step >= len(self.dates) - 1)
        
        info = {'portfolio_return': portfolio_return, 'turnover': turnover}
        return self._get_obs(), float(reward), done, False, info


# Evaluation


In [ ]:
def evaluate_on_validation(model, val_df, algo_name):
    """Run a trained model on the full validation period and compute metrics."""
    
    env = PortfolioEnv(val_df, episode_length=len(val_df['date'].unique()) - 2, mode='eval')
    
    obs, _ = env.reset()
    daily_returns = []
    turnovers = []
    
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        daily_returns.append(info['portfolio_return'])
        turnovers.append(info['turnover'])
        done = done or truncated
    
    daily_returns = np.array(daily_returns)
    
    # Compute metrics
    cumulative = np.prod(1 + daily_returns) - 1
    n_days = len(daily_returns)
    ann_return = (1 + cumulative) ** (252 / max(n_days, 1)) - 1
    ann_vol = np.std(daily_returns) * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    downside = daily_returns[daily_returns < 0]
    downside_vol = np.std(downside) * np.sqrt(252) if len(downside) > 0 else 0.001
    sortino = ann_return / downside_vol if downside_vol > 0 else 0
    
    cum_returns = np.cumprod(1 + daily_returns)
    running_max = np.maximum.accumulate(cum_returns)
    drawdowns = (cum_returns - running_max) / running_max
    max_drawdown = np.min(drawdowns)
    
    calmar = ann_return / abs(max_drawdown) if abs(max_drawdown) > 0 else 0
    win_rate = np.mean(daily_returns > 0)
    mean_turnover = np.mean(turnovers)
    
    result = {
        'cumulative_return': round(cumulative * 100, 2),
        'ann_return': round(ann_return * 100, 2),
        'ann_volatility': round(ann_vol * 100, 2),
        'sharpe': round(sharpe, 4),
        'sortino': round(sortino, 4),
        'max_drawdown': round(max_drawdown * 100, 2),
        'calmar': round(calmar, 4),
        'win_rate': round(win_rate * 100, 2),
        'mean_daily_turnover': round(mean_turnover, 4),
        'n_trading_days': n_days,
    }
    
    return result


def evaluate_model(model, test_df, algo_name):
    """Run model on full test period; returns (result_dict, daily_returns, weights)."""
    n_test_days = len(test_df['date'].unique())
    env = PortfolioEnv(test_df, episode_length=n_test_days - 2, mode='eval')

    obs, _ = env.reset()
    daily_returns = []
    daily_weights = []
    turnovers = []

    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        daily_returns.append(info['portfolio_return'])
        daily_weights.append(env.weights.copy())
        turnovers.append(info['turnover'])
        done = done or truncated

    daily_returns = np.array(daily_returns)

    # Metrics
    cumulative = np.prod(1 + daily_returns) - 1
    n_days = len(daily_returns)
    ann_return = (1 + cumulative) ** (252 / max(n_days, 1)) - 1
    ann_vol = np.std(daily_returns) * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0

    downside = daily_returns[daily_returns < 0]
    downside_vol = np.std(downside) * np.sqrt(252) if len(downside) > 0 else 0.001
    sortino = ann_return / downside_vol if downside_vol > 0 else 0

    cum_returns = np.cumprod(1 + daily_returns)
    running_max = np.maximum.accumulate(cum_returns)
    drawdowns = (cum_returns - running_max) / running_max
    max_drawdown = np.min(drawdowns)

    calmar = ann_return / abs(max_drawdown) if abs(max_drawdown) > 0 else 0
    win_rate = np.mean(daily_returns > 0)
    mean_turnover = np.mean(turnovers)

    result = {
        'cumulative_return': round(cumulative * 100, 2),
        'ann_return': round(ann_return * 100, 2),
        'ann_volatility': round(ann_vol * 100, 2),
        'sharpe': round(sharpe, 4),
        'sortino': round(sortino, 4),
        'max_drawdown': round(max_drawdown * 100, 2),
        'calmar': round(calmar, 4),
        'win_rate': round(win_rate * 100, 2),
        'mean_daily_turnover': round(mean_turnover, 4),
        'n_trading_days': n_days,
    }

    return result, daily_returns, np.array(daily_weights)


# Baselines


In [ ]:
def run_baselines(val_df):
    """Compute baselines (sprint4_running.py version — used for the validation grid)."""
    dates = sorted(val_df['date'].unique())
    stocks = sorted(val_df['PERMNO'].unique())
    n_assets = len(stocks)
    
    stock_to_idx = {s: i for i, s in enumerate(stocks)}
    ret_col = 'raw_RET' if 'raw_RET' in val_df.columns else 'RET'
    ret_matrix = np.zeros((len(dates), n_assets))
    
    for _, row in val_df.iterrows():
        d_idx = dates.index(row['date'])
        s_idx = stock_to_idx[row['PERMNO']]
        ret_matrix[d_idx, s_idx] = row[ret_col]
    
    baselines = {}
    
    # Buy-and-Hold
    weights = np.ones(n_assets) / n_assets
    daily_rets = []
    for t in range(1, len(dates)):
        r = np.dot(weights, ret_matrix[t])
        daily_rets.append(r)
        weights = weights * (1 + ret_matrix[t])
        weights /= weights.sum()
    baselines['Buy_Hold'] = np.array(daily_rets)
    
    # Equal-Weight
    weights = np.ones(n_assets) / n_assets
    daily_rets = []
    for t in range(1, len(dates)):
        r = np.dot(weights, ret_matrix[t])
        daily_rets.append(r)
    baselines['Equal_Weight'] = np.array(daily_rets)
    
    # Market-Cap Weighted
    mkt_col = 'mkt_cap' if 'mkt_cap' in val_df.columns else None
    if mkt_col:
        daily_rets = []
        for t in range(1, len(dates)):
            date = dates[t]
            day_data = val_df[val_df['date'] == date].sort_values('PERMNO')
            if len(day_data) == n_assets:
                mc = day_data[mkt_col].values.astype(float)
                mc = np.maximum(mc, 0)
                w = mc / mc.sum() if mc.sum() > 0 else np.ones(n_assets) / n_assets
            else:
                w = np.ones(n_assets) / n_assets
            r = np.dot(w, ret_matrix[t])
            daily_rets.append(r)
        baselines['MktCap_Weight'] = np.array(daily_rets)
    
    results = {}
    for name, rets in baselines.items():
        cumulative = np.prod(1 + rets) - 1
        n_days = len(rets)
        ann_return = (1 + cumulative) ** (252 / max(n_days, 1)) - 1
        ann_vol = np.std(rets) * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0
        downside = rets[rets < 0]
        downside_vol = np.std(downside) * np.sqrt(252) if len(downside) > 0 else 0.001
        sortino = ann_return / downside_vol
        cum_rets = np.cumprod(1 + rets)
        running_max = np.maximum.accumulate(cum_rets)
        drawdowns = (cum_rets - running_max) / running_max
        max_dd = np.min(drawdowns)
        calmar = ann_return / abs(max_dd) if abs(max_dd) > 0 else 0
        
        results[name] = {
            'cumulative_return': round(cumulative * 100, 2),
            'ann_return': round(ann_return * 100, 2),
            'ann_volatility': round(ann_vol * 100, 2),
            'sharpe': round(sharpe, 4),
            'sortino': round(sortino, 4),
            'max_drawdown': round(max_dd * 100, 2),
            'calmar': round(calmar, 4),
            'win_rate': round(np.mean(rets > 0) * 100, 2),
            'mean_daily_turnover': 0,
            'n_trading_days': n_days,
        }
    
    return results


def run_baselines_with_daily(test_df):
    """Compute baselines (sprint4_test_evaluation.py version — used for val + test eval).
    Returns (results_dict, daily_returns_dict)."""
    dates = sorted(test_df['date'].unique())
    stocks = sorted(test_df['PERMNO'].unique())
    n_assets = len(stocks)

    stock_to_idx = {s: i for i, s in enumerate(stocks)}
    ret_col = 'raw_RET' if 'raw_RET' in test_df.columns else 'RET'
    ret_matrix = np.zeros((len(dates), n_assets))

    for _, row in test_df.iterrows():
        d_idx = dates.index(row['date'])
        s_idx = stock_to_idx[row['PERMNO']]
        ret_matrix[d_idx, s_idx] = row[ret_col]

    baselines = {}

    # Buy-and-Hold
    weights = np.ones(n_assets) / n_assets
    daily_rets = []
    for t in range(1, len(dates)):
        r = np.dot(weights, ret_matrix[t])
        daily_rets.append(r)
        weights = weights * (1 + ret_matrix[t])
        weights /= weights.sum()
    baselines['Buy_Hold'] = np.array(daily_rets)

    # Equal-Weight
    weights = np.ones(n_assets) / n_assets
    daily_rets = []
    for t in range(1, len(dates)):
        r = np.dot(weights, ret_matrix[t])
        daily_rets.append(r)
    baselines['Equal_Weight'] = np.array(daily_rets)

    # Market-Cap Weighted
    mkt_col = 'MKT_CAP' if 'MKT_CAP' in test_df.columns else ('mkt_cap' if 'mkt_cap' in test_df.columns else None)
    if mkt_col:
        daily_rets = []
        for t in range(1, len(dates)):
            date = dates[t]
            day_data = test_df[test_df['date'] == date].sort_values('PERMNO')
            if len(day_data) == n_assets:
                mc = np.nan_to_num(day_data[mkt_col].values.astype(float), nan=0.0)
                mc = np.maximum(mc, 0)
                w = mc / mc.sum() if mc.sum() > 0 else np.ones(n_assets) / n_assets
            else:
                w = np.ones(n_assets) / n_assets
            r = np.dot(w, ret_matrix[t])
            daily_rets.append(r)
        baselines['MktCap_Weight'] = np.array(daily_rets)

    results = {}
    all_daily = {}
    for name, rets in baselines.items():
        cumulative = np.prod(1 + rets) - 1
        n_days = len(rets)
        ann_return = (1 + cumulative) ** (252 / max(n_days, 1)) - 1
        ann_vol = np.std(rets) * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0
        downside = rets[rets < 0]
        downside_vol = np.std(downside) * np.sqrt(252) if len(downside) > 0 else 0.001
        sortino = ann_return / downside_vol
        cum_rets = np.cumprod(1 + rets)
        running_max = np.maximum.accumulate(cum_rets)
        dd = (cum_rets - running_max) / running_max
        max_dd = np.min(dd)
        calmar = ann_return / abs(max_dd) if abs(max_dd) > 0 else 0

        results[name] = {
            'cumulative_return': round(cumulative * 100, 2),
            'ann_return': round(ann_return * 100, 2),
            'ann_volatility': round(ann_vol * 100, 2),
            'sharpe': round(sharpe, 4),
            'sortino': round(sortino, 4),
            'max_drawdown': round(max_dd * 100, 2),
            'calmar': round(calmar, 4),
            'win_rate': round(np.mean(rets > 0) * 100, 2),
            'mean_daily_turnover': 0,
            'n_trading_days': n_days,
        }
        all_daily[name] = rets

    return results, all_daily


# Data diagnostics


In [ ]:
def diagnose_data(df, label="Data"):
    """Check for NaN/Inf in data before training."""
    print(f"\n  --- {label} Diagnostics ---")
    
    feature_cols = ["PRC","VOL","RET","SHROUT","OPENPRC","vix_index",
                    "RSI","MACD","BB_PCT","ATR","rolling_vol_60",
                    "SMA_20","SMA_50","log_vol"]
    
    for col in feature_cols:
        n_nan = df[col].isna().sum()
        n_inf = np.isinf(df[col].astype(float)).sum() if df[col].dtype in ['float64', 'float32'] else 0
        if n_nan > 0 or n_inf > 0:
            print(f"    {col}: {n_nan} NaN, {n_inf} Inf")
    
    # Overall stats
    numeric_df = df[feature_cols].select_dtypes(include=[np.number])
    total_nan = numeric_df.isna().sum().sum()
    total_inf = np.isinf(numeric_df.values).sum() if numeric_df.shape[0] > 0 else 0
    print(f"  Total: {total_nan} NaN, {total_inf} Inf across {len(feature_cols)} features")
    
    # Check value ranges
    for col in feature_cols:
        if df[col].dtype in ['float64', 'float32', 'int64']:
            max_abs = df[col].abs().max()
            if max_abs > 100:
                print(f"    {col}: max abs value = {max_abs:.1f} (may cause instability)")


# Load data


In [ ]:
train_df = pd.read_csv(TRAIN_DATA_PATH, parse_dates=['date'])
val_df = pd.read_csv(VAL_DATA_PATH, parse_dates=['date'])

# Ensure raw_RET exists
if 'raw_RET' not in train_df.columns:
    if os.path.exists(SCALER_PARAMS_PATH):
        scaler = pd.read_csv(SCALER_PARAMS_PATH, index_col=0)
        ret_mean = scaler.loc['RET', 'mean']
        ret_std = scaler.loc['RET', 'std']
        train_df['raw_RET'] = train_df['RET'] * ret_std + ret_mean
        val_df['raw_RET'] = val_df['RET'] * ret_std + ret_mean
        print("  Denormalized RET using scaler_params.csv")
    else:
        print("  WARNING: No raw_RET and no scaler_params.csv")

# Filter to common stocks
train_dates = sorted(train_df['date'].unique())
val_dates = sorted(val_df['date'].unique())

train_coverage = train_df.groupby('PERMNO')['date'].nunique()
full_train_stocks = set(train_coverage[train_coverage == len(train_dates)].index)

val_coverage = val_df.groupby('PERMNO')['date'].nunique()
full_val_stocks = set(val_coverage[val_coverage == len(val_dates)].index)

common_stocks = full_train_stocks & full_val_stocks
train_df = train_df[train_df['PERMNO'].isin(common_stocks)].copy()
val_df = val_df[val_df['PERMNO'].isin(common_stocks)].copy()

n_stocks = len(common_stocks)
print(f"  Train: {len(train_df)} rows, {n_stocks} stocks, {len(train_dates)} days")
print(f"  Val:   {len(val_df)} rows, {n_stocks} stocks, {len(val_dates)} days")

# Run data diagnostics
diagnose_data(train_df, "Training Data")
diagnose_data(val_df, "Validation Data")


# Run baselines


In [ ]:
baseline_results = run_baselines(val_df)
for name, metrics in baseline_results.items():
    print(f"  {name}: Cum={metrics['cumulative_return']:.1f}%, Sharpe={metrics['sharpe']:.3f}")

all_results = []

for name, metrics in baseline_results.items():
    row = {
        'algorithm': name, 'timesteps': 'N/A', 'learning_rate': 'N/A',
        'seed': 'N/A', 'train_time_sec': 0, **metrics
    }
    all_results.append(row)


# PPO grid


In [ ]:
# Build PPO env once for all PPO experiments (saves time)
ppo_train_env_base = PortfolioEnv(train_df, episode_length=EPISODE_LENGTH, mode='train')

for ts, lr, seed in product(TIMESTEP_GRID, PPO_LR_GRID, SEEDS):
    exp_name = f"PPO_ts{ts}_lr{lr}_s{seed}"
    
    try:
        # Create a fresh env wrapper each time (reuses the clean data)
        def make_ppo_env():
            env = PortfolioEnv.__new__(PortfolioEnv)
            env.__dict__.update(ppo_train_env_base.__dict__.copy())
            env.mode = 'train'
            env.weights = np.ones(env.n_assets) / env.n_assets
            env.current_step = 0
            env.start_step = 0
            return env
        
        env = DummyVecEnv([make_ppo_env])
        
        t0 = time.time()
        model = PPO(
            "MlpPolicy", env, verbose=0,
            learning_rate=lr,
            n_steps=2048,
            batch_size=128,
            n_epochs=10,
            gamma=0.99,
            ent_coef=0.01,
            seed=seed
        )
        model.learn(total_timesteps=ts)
        train_time = time.time() - t0
        
        model_path = os.path.join(MODEL_DIR, exp_name)
        model.save(model_path)
        
        metrics = evaluate_on_validation(model, val_df, 'PPO')
        
        row = {
            'algorithm': 'PPO', 'timesteps': ts, 'learning_rate': lr,
            'seed': seed, 'train_time_sec': round(train_time, 1), **metrics
        }
        all_results.append(row)
        
        print(f"  {exp_name}: Cum={metrics['cumulative_return']:.1f}%, Sharpe={metrics['sharpe']:.3f}, "
              f"MaxDD={metrics['max_drawdown']:.1f}%, Time={train_time:.0f}s")
        
        del model, env
        
    except Exception as e:
        print(f"  ERROR: {e}")
        all_results.append({
            'algorithm': 'PPO', 'timesteps': ts, 'learning_rate': lr,
            'seed': seed, 'train_time_sec': 0, 'cumulative_return': None,
            'sharpe': None, 'error': str(e)
        })
    
    pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)


# Best PPO config


In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_CSV, index=False)

# Best config per algorithm — PPO
algo_df = results_df[results_df['algorithm'] == 'PPO'].dropna(subset=['sharpe'])

algo_df = algo_df.copy()
algo_df['sharpe'] = pd.to_numeric(algo_df['sharpe'], errors='coerce')
algo_df['cumulative_return'] = pd.to_numeric(algo_df['cumulative_return'], errors='coerce')
algo_df['max_drawdown'] = pd.to_numeric(algo_df['max_drawdown'], errors='coerce')
algo_df['train_time_sec'] = pd.to_numeric(algo_df['train_time_sec'], errors='coerce')

grouped = algo_df.groupby(['timesteps', 'learning_rate']).agg({
    'sharpe': ['mean', 'std'],
    'cumulative_return': 'mean',
    'max_drawdown': 'mean',
    'train_time_sec': 'mean',
}).reset_index()

grouped.columns = ['timesteps', 'learning_rate',
                  'mean_sharpe', 'std_sharpe',
                  'mean_cum_return', 'mean_max_dd', 'mean_train_time']

best_idx = grouped['mean_sharpe'].idxmax()
best = grouped.loc[best_idx]

best_ppo = {
    'timesteps': int(best['timesteps']),
    'learning_rate': float(best['learning_rate']),
    'mean_sharpe': round(float(best['mean_sharpe']), 4),
    'std_sharpe': round(float(best['std_sharpe']), 4),
    'mean_cumulative_return': round(float(best['mean_cum_return']), 2),
    'mean_max_drawdown': round(float(best['mean_max_dd']), 2),
}

print(f"  Best PPO: ts={best['timesteps']:.0f}, lr={best['learning_rate']}, "
      f"Sharpe={best['mean_sharpe']:.3f} ± {best['std_sharpe']:.3f}")

print("\n  Baselines:")
for name, metrics in baseline_results.items():
    print(f"    {name}: Sharpe={metrics['sharpe']:.3f}")

# Save best config — read existing JSON if present and update PPO entry, leaving DQN intact
best_configs_path = 'results/pipeline_a_best_configs.json'
if os.path.exists(best_configs_path):
    with open(best_configs_path, 'r') as f:
        best_configs = json.load(f)
else:
    best_configs = {}
best_configs['PPO'] = best_ppo
with open(best_configs_path, 'w') as f:
    json.dump(best_configs, f, indent=2)

# Seed stability
stability = algo_df.groupby(['timesteps', 'learning_rate']).agg({
    'sharpe': ['mean', 'std', 'min', 'max'],
    'cumulative_return': ['mean', 'std'],
})

print(f"\n  PPO — Sharpe across seeds:")
for (ts, lr), row in stability.iterrows():
    mean_s = row[('sharpe', 'mean')]
    std_s = row[('sharpe', 'std')]
    min_s = row[('sharpe', 'min')]
    max_s = row[('sharpe', 'max')]
    print(f"    ts={ts:>10}, lr={lr}: "
          f"Sharpe {mean_s:.3f} ± {std_s:.3f} [{min_s:.3f}, {max_s:.3f}]")


# Test evaluation


In [ ]:
test_df = pd.read_csv(TEST_DATA_PATH, parse_dates=['date'])

# Reuse the 47-stock common_stocks set from the train+val load above
test_df = test_df[test_df['PERMNO'].isin(common_stocks)].copy()

# Denormalize RET to get raw returns for rewards
if 'raw_RET' not in test_df.columns:
    if os.path.exists(SCALER_PARAMS_PATH):
        scaler = pd.read_csv(SCALER_PARAMS_PATH, index_col=0)
        ret_mean = scaler.loc['RET', 'mean']
        ret_std = scaler.loc['RET', 'std']
        test_df['raw_RET'] = test_df['RET'] * ret_std + ret_mean
        print(f"  Denormalized RET using scaler_params.csv (mean={ret_mean:.6f}, std={ret_std:.6f})")
    else:
        print("  WARNING: No raw_RET and no scaler_params.csv — using normalized RET for rewards")

print(f"  Val:  {len(val_df)} rows, {val_df['PERMNO'].nunique()} stocks, "
      f"{val_df['date'].nunique()} days ({val_df['date'].min().date()} to {val_df['date'].max().date()})")
print(f"  Test: {len(test_df)} rows, {test_df['PERMNO'].nunique()} stocks, "
      f"{test_df['date'].nunique()} days ({test_df['date'].min().date()} to {test_df['date'].max().date()})")

# Check test coverage — warn if any stock has gaps
test_dates_all = sorted(test_df['date'].unique())
test_cov = test_df.groupby('PERMNO')['date'].nunique()
partial = test_cov[test_cov < len(test_dates_all)]
if len(partial) > 0:
    print(f"  WARNING: {len(partial)} stocks have partial test coverage (missing dates filled with 0)")

assert val_df['PERMNO'].nunique() == 47, f"Expected 47 val stocks, got {val_df['PERMNO'].nunique()}"
assert test_df['PERMNO'].nunique() == 47, f"Expected 47 test stocks, got {test_df['PERMNO'].nunique()}"

# Use the just-computed best PPO config from the grid above
BEST_PPO = {'timesteps': best_ppo['timesteps'],
            'learning_rate': best_ppo['learning_rate'],
            'seeds': SEEDS}

# Evaluate on BOTH validation and test
for dataset_label, eval_df in [("VALIDATION (2019-2021)", val_df), ("TEST (2022-2024)", test_df)]:
    print(f"\n  EVALUATING ON: {dataset_label}")

    # Baselines on this dataset
    baseline_results, baseline_daily = run_baselines_with_daily(eval_df)
    for name, metrics in baseline_results.items():
        print(f"    {name}: Cum={metrics['cumulative_return']}%, Sharpe={metrics['sharpe']}, MaxDD={metrics['max_drawdown']}%")

    all_results = []
    all_daily_returns = dict(baseline_daily)

    for name, metrics in baseline_results.items():
        row = {'dataset': dataset_label, 'algorithm': name, 'timesteps': None,
               'learning_rate': None, 'seed': None}
        row.update(metrics)
        all_results.append(row)

    # PPO at best config
    print(f"\n  --- PPO (ts={BEST_PPO['timesteps']}, lr={BEST_PPO['learning_rate']}) ---")
    for seed in BEST_PPO['seeds']:
        model_path = os.path.join(MODEL_DIR, f"PPO_ts{BEST_PPO['timesteps']}_lr{BEST_PPO['learning_rate']}_s{seed}")
        if not (os.path.exists(model_path) or os.path.exists(model_path + '.zip') or os.path.isdir(model_path)):
            print(f"  WARNING: Model not found: {model_path}")
            continue
        model = PPO.load(model_path)
        metrics, daily_rets, weights = evaluate_model(model, eval_df, 'PPO')
        print(f"    Seed {seed}: Cum={metrics['cumulative_return']}%, Sharpe={metrics['sharpe']}, "
              f"MaxDD={metrics['max_drawdown']}%, Turnover={metrics['mean_daily_turnover']}")
        row = {'dataset': dataset_label, 'algorithm': 'PPO', 'timesteps': BEST_PPO['timesteps'],
               'learning_rate': BEST_PPO['learning_rate'], 'seed': seed}
        row.update(metrics)
        all_results.append(row)
        all_daily_returns[f"PPO_s{seed}"] = daily_rets

    # Save per-dataset results
    tag = "val" if "VAL" in dataset_label else "test"
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(f'results/pipeline_a_{tag}_results.csv', index=False)

    max_len = max(len(v) for v in all_daily_returns.values())
    daily_df = pd.DataFrame({k: np.pad(v, (0, max_len - len(v)), constant_values=np.nan)
                             for k, v in all_daily_returns.items()})
    daily_df.to_csv(f'results/pipeline_a_{tag}_daily_returns.csv', index=False)

    # Summary table
    print(f"\n  {dataset_label} — SUMMARY")
    print(f"  {'Algorithm':<15} {'Cum%':>8} {'Sharpe':>8} {'MaxDD%':>8} {'Sortino':>8} {'Turnover':>10}")
    print("  " + "-" * 60)
    for _, row in results_df.iterrows():
        algo = row['algorithm']
        if pd.notna(row.get('seed')):
            algo = f"{algo}_s{int(row['seed'])}"
        print(f"  {algo:<15} {row['cumulative_return']:>8.1f} {row['sharpe']:>8.4f} "
              f"{row['max_drawdown']:>8.1f} {row['sortino']:>8.4f} {row['mean_daily_turnover']:>10.4f}")

    print("  " + "-" * 60)
    algo_rows = results_df[results_df['algorithm'] == 'PPO']
    if len(algo_rows) > 0:
        print(f"  {'PPO_mean':<15} {algo_rows['cumulative_return'].mean():>8.1f} "
              f"{algo_rows['sharpe'].mean():>8.4f} {algo_rows['max_drawdown'].mean():>8.1f} "
              f"{algo_rows['sortino'].mean():>8.4f} {algo_rows['mean_daily_turnover'].mean():>10.4f}")
        print(f"  {'PPO_std':<15} {algo_rows['cumulative_return'].std():>8.1f} "
              f"{algo_rows['sharpe'].std():>8.4f} {algo_rows['max_drawdown'].std():>8.1f} "
              f"{algo_rows['sortino'].std():>8.4f} {algo_rows['mean_daily_turnover'].std():>10.4f}")
